In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Part 2: Classification

Consider a simulated dataset generated as follows:
1. Set the seed: `set.seed(42)`.
2. For each data point $i$, sample its label from a Bernoulli distribution $y_i \sim B(p)$.
3. Depending on the label $y_i \in \{0,1\}$, sample the data point $x_i$ as follows:
   - $y_i = 0 \Rightarrow x_i \sim 0.5 \mathcal{N}(\mu_0^{(a)}, C_0^{(a)}) + 0.5 \mathcal{N}(\mu_0^{(b)}, C_0^{(b)})$
   - $y_i = 1 \Rightarrow x_i \sim \mathcal{N}(\mu_1, C_1)$

where $\mu_0^{(a)} = [0, 1]^\top, \mu_0^{(b)} = [0, -1]^\top, \mu_1 = [\varepsilon, 0]^\top$, and $C_0^{(a)} = C_0^{(b)} = 0.5 I_2, C_1 = I_2$.

**Q1.** Consider $p = 0.5, \varepsilon = 2$. Simulate Dtrain = D(200 | 2, 0.5) and Dtest = D(1000 | 2, 0.5).
- (a) What is the mathematical expression for the optimal Bayes classifier in this setting? And for its boundary region?
- (b) Plot the boundary region for the Bayes classifier overlaid with the scattered data points of Dtrain. Use different colors for each class and use `contour` for plotting the boundary region.
- (c) Estimate the error of the Bayes classifier on the samples from Dtest.
- (d) Train a LDA and a Logistic Regression classifier on Dtrain and estimate their errors on Dtest. How do these errors compare to (c)? Comment your results.

**Q2.** Consider $p = 0.5$ fixed and $\varepsilon$ varying. Simulate 51 datasets $D^{(i)}$train = D(200 | $\varepsilon_i$, 0.5) and $D^{(i)}$test = D(1000 | $\varepsilon_i$, 0.5) with $\varepsilon_i = 1 + i/10$ and $i = 0, \dots, 50$.
- (a) Calculate the test error for the Bayes classifier, LDA, and Logistic Regression for each dataset (train on $D^{(i)}$train, test on $D^{(i)}$test).
- (b) Plot a curve showing the error with each classifier as a function of $\varepsilon$. Comment your results. You should notice that for large $\varepsilon$ the logistic regression throws an error — can you explain what is happening? What would be a good approach for limiting this problem?

In [ ]:
df = pd.read_csv('../datasets/Caschool.csv', sep=';')
print('Shape:', df.shape)
display(df.head(5))

In [ ]:
# Create binary target: 1 if testscr >= median, 0 otherwise
median_score = df['testscr'].median()
df['high_score'] = (df['testscr'] >= median_score).astype(int)
print(f'Median test score: {median_score:.2f}')
print(df['high_score'].value_counts())

In [ ]:
feature_cols = ['str', 'avginc', 'elpct', 'mealpct', 'expnstu', 'calwpct']
X = df[feature_cols].dropna()
y = df.loc[X.index, 'high_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

In [ ]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)
lda_acc = accuracy_score(y_test, lda.predict(X_test))
print(f'LDA accuracy: {lda_acc:.4f}')

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_test))
print(f'Logistic Regression accuracy: {lr_acc:.4f}')

In [ ]:
# Decision boundary using two features: str (student-teacher ratio) and avginc
feat1, feat2 = 'str', 'avginc'
X2_train = X_train[[feat1, feat2]].values
X2_test  = X_test[[feat1, feat2]].values

lda2 = LinearDiscriminantAnalysis().fit(X2_train, y_train)

x_min, x_max = X[[feat1]].values.min() - 1, X[[feat1]].values.max() + 1
y_min, y_max = X[[feat2]].values.min() - 1, X[[feat2]].values.max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = lda2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
scatter = ax.scatter(X_test[feat1], X_test[feat2],
                     c=y_test, cmap='RdBu', edgecolors='k',
                     linewidths=0.4, alpha=0.8)
ax.set_xlabel(feat1)
ax.set_ylabel(feat2)
ax.set_title('LDA decision boundary\n(two features, test set)')
plt.colorbar(scatter, ax=ax, label='high_score')
plt.tight_layout()
plt.show()

## Your answers here
